# Day 046 — Exercise 4: Change Analysis

**What you'll build:** `period_changes(series) -> pd.DataFrame` — compute absolute change (`diff()`), percentage change (`pct_change()`), and cumulative sum (`cumsum()`) alongside the original values, all in one DataFrame.

**Why it matters:** Absolute values answer 'what is it?'. Change metrics answer 'is it getting better or worse?'. Percentage change normalises across different scales. Cumulative sum tracks total throughput over time.

## Provided: Setup + parse_time_series + rolling functions

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

def make_sample_ts(n_days: int = 90, seed: int = 42) -> pd.DataFrame:
    """Return a reproducible daily time-series DataFrame for exercises."""
    rng   = np.random.default_rng(seed)
    dates = pd.date_range('2024-01-01', periods=n_days, freq='D')
    vals  = 1000.0 + (rng.standard_normal(n_days).cumsum() * 50)
    return pd.DataFrame({'date': dates.strftime('%Y-%m-%d'),
                         'value': vals.round(2)})


import pandas as pd
import warnings
warnings.filterwarnings('ignore')

def parse_time_series(df: pd.DataFrame, date_col: str) -> pd.DataFrame:
    df = df.copy()
    df[date_col] = pd.to_datetime(df[date_col], errors='coerce')
    df = df.dropna(subset=[date_col])
    df = df.set_index(date_col).sort_index()
    return df

def date_features(df: pd.DataFrame) -> pd.DataFrame:
    idx = df.index
    return pd.DataFrame({
        'year':        idx.year,
        'month':       idx.month,
        'day':         idx.day,
        'day_of_week': idx.dayofweek,
        'quarter':     idx.quarter,
    }, index=idx)


def resample_series(series: pd.Series, freq: str, agg: str = 'sum') -> pd.Series:
    return series.resample(freq).agg(agg)

def multi_freq_summary(series: pd.Series) -> dict:
    return {
        'daily':  resample_series(series, 'D'),
        'weekly': resample_series(series, 'W'),
    }


def rolling_mean(series: pd.Series, window: int, min_periods: int = 1) -> pd.Series:
    return series.rolling(window=window, min_periods=min_periods).mean()

def rolling_stats(series: pd.Series, window: int) -> pd.DataFrame:
    r = series.rolling(window=window, min_periods=1)
    return pd.DataFrame({
        'mean': r.mean(),
        'std':  r.std(),
        'min':  r.min(),
        'max':  r.max(),
    })

## Your Implementation

In [ ]:
def period_changes(series: pd.Series) -> pd.DataFrame:
    """
    Return a DataFrame with value, change, pct_change, and cumulative columns.

    Columns:
        value:      original values
        change:     series.diff()  — absolute change vs previous period (NaN for first row)
        pct_change: series.pct_change() * 100  — % change (NaN for first row)
        cumulative: series.cumsum()  — running total
    """
    # TODO: return pd.DataFrame({
    #     'value':      series,
    #     'change':     series.diff(),
    #     'pct_change': series.pct_change() * 100,
    #     'cumulative': series.cumsum(),
    # })
    pass

## Check Your Work

In [ ]:
def _run_checks():
    total = 5
    passed = 0

    df_raw = make_sample_ts(20)
    series = parse_time_series(df_raw, 'date')['value']

    # Check 1: period_changes defined, returns DataFrame
    try:
        assert 'period_changes' in globals()
        pc = period_changes(series)
        assert isinstance(pc, pd.DataFrame), \
            f'expected DataFrame, got {type(pc).__name__}'
        passed += 1; print('\u2705 Check 1: period_changes returns DataFrame')
    except Exception as e:
        print(f'\u274c Check 1: {e}')
        print(f'\nScore: {passed}/{total}'); return

    # Check 2: has all four columns
    try:
        for col in ('value', 'change', 'pct_change', 'cumulative'):
            assert col in pc.columns, f'missing column: {col}'
        passed += 1; print('\u2705 Check 2: has value, change, pct_change, cumulative')
    except Exception as e:
        print(f'\u274c Check 2: {e}')

    # Check 3: first row change is NaN (no previous period)
    try:
        assert pd.isna(pc['change'].iloc[0]), \
            f'first change should be NaN, got {pc["change"].iloc[0]}'
        assert pd.isna(pc['pct_change'].iloc[0]), \
            f'first pct_change should be NaN, got {pc["pct_change"].iloc[0]}'
        passed += 1; print('\u2705 Check 3: first row change and pct_change are NaN')
    except Exception as e:
        print(f'\u274c Check 3: {e}')

    # Check 4: value column matches original series
    try:
        assert (pc['value'] == series).all(), \
            'value column should equal the input series'
        passed += 1; print('\u2705 Check 4: value column matches original series')
    except Exception as e:
        print(f'\u274c Check 4: {e}')

    # Check 5: cumulative is correct (cumsum)
    try:
        expected_cumsum = series.cumsum()
        diff = (pc['cumulative'] - expected_cumsum).abs().max()
        assert diff < 1e-9, f'cumulative != series.cumsum() (max diff={diff:.2e})'
        passed += 1; print('\u2705 Check 5: cumulative == series.cumsum()')
    except Exception as e:
        print(f'\u274c Check 5: {e}')

    if passed == total:
        print('\U0001f389 Exercise complete!')
    print(f'\nScore: {passed}/{total}')


_run_checks()

## Solution

<details>
<summary>Click to reveal</summary>

```python
def period_changes(series: pd.Series) -> pd.DataFrame:
    return pd.DataFrame({
        'value':      series,
        'change':     series.diff(),
        'pct_change': series.pct_change() * 100,
        'cumulative': series.cumsum(),
    })
```

</details>